In [1]:
from pathlib import Path
import pandas as pd
import re

In [ ]:
ROOT = Path(r"//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection")

rows = []

for week_dir in ROOT.glob("week_*"):
    if not week_dir.is_dir():
        continue

    for box_dir in week_dir.iterdir():
        if not box_dir.is_dir():
            continue

        box_name = box_dir.name

        for rec_dir in box_dir.iterdir():
            if not rec_dir.is_dir():
                continue

            avi_files = [
                p for p in rec_dir.iterdir()
                if p.is_file() and p.suffix.lower() == ".avi"
            ]

            raw_files = [
                p for p in rec_dir.iterdir()
                if p.is_file() and p.suffix.lower() == ".raw"
            ]

            avi_bytes = sum(f.stat().st_size for f in avi_files)
            raw_bytes = sum(f.stat().st_size for f in raw_files)

            rows.append({
                "week": week_dir.name,
                "box": box_name,
                "recording": rec_dir.name,
                "n_avi_files": len(avi_files),
                "n_raw_files": len(raw_files),
                "avi_gb": avi_bytes / 1024**3,
                "raw_gb": raw_bytes / 1024**3,
            })

df = pd.DataFrame(rows)

In [22]:
df

,week,box,recording,n_avi_files,n_raw_files,avi_gb,raw_gb
0,week_31,box_1,2024_07_30_16_11_37_recordings,1,1,191.185226,0.051439
1,week_31,box_1,2024_07_31_16_54_30_recordings,1,1,143.287217,3.328046
2,week_31,box_1,2024_08_01_16_26_38_recordings,1,1,148.822420,4.167566
3,week_31,box_2,2024_07_30_16_23_44_recordings,1,1,194.936035,6.030994
4,week_31,box_2,2024_07_31_17_03_41_recordings,1,1,183.034245,6.900410
...,...,...,...,...,...,...,...
60,week_38,box_2,2024_09_24_14_23_34_recordings,1,1,185.334309,19.681995
61,week_38,box_2,2024_09_25_15_54_54_recordings,1,1,144.671875,22.499619
62,week_38,box_3,2024_09_24_14_22_25_recordings,1,1,195.011711,15.758220
63,week_38,box_3,2024_09_25_15_56_11_recordings,1,1,127.202480,4.297142


In [24]:
summary_box = (
    df.groupby(["week", "box"], as_index=False)
      .agg(
          recordings=("recording", "count"),
          avi_files=("n_avi_files", "sum"),
          raw_files=("n_raw_files", "sum"),
          avi_GB=("avi_gb", "sum"),
          raw_GB=("raw_gb", "sum"),
      )
)

summary_box["avi_GB"] = summary_box["avi_GB"].round(0).map(lambda x: f"{int(x):,}")
summary_box["raw_GB"] = summary_box["raw_GB"].round(0).map(lambda x: f"{int(x):,}")

summary_box

,week,box,recordings,avi_files,raw_files,avi_GB,raw_GB
0,week_31,box_1,3,3,3,483,8
1,week_31,box_2,3,3,3,543,20
2,week_31,box_3,3,3,3,560,15
3,week_32,box_1,4,4,4,651,23
4,week_32,box_3,4,4,4,703,48
5,week_33,box_1,2,2,2,344,24
6,week_33,box_2,3,3,3,412,38
7,week_33,box_3,3,3,3,465,59
8,week_34,box_1,3,3,3,536,38
9,week_34,box_2,3,3,3,508,65


In [25]:
summary_week = (
    df.groupby("week", as_index=False)
      .agg(
          recordings=("recording", "count"),
          avi_files=("n_avi_files", "sum"),
          raw_files=("n_raw_files", "sum"),
          avi_GB=("avi_gb", "sum"),
          raw_GB=("raw_gb", "sum"),
      )
)

summary_week["avi_GB"] = summary_week["avi_GB"].round(0).map(lambda x: f"{int(x):,}")
summary_week["raw_GB"] = summary_week["raw_GB"].round(0).map(lambda x: f"{int(x):,}")

summary_week

,week,recordings,avi_files,raw_files,avi_GB,raw_GB
0,week_31,9,9,9,"1,586",42
1,week_32,8,8,8,"1,355",71
2,week_33,8,8,8,"1,221",121
3,week_34,9,9,9,"1,601",171
4,week_35,9,9,9,"1,675",102
5,week_36,9,9,9,"1,545",129
6,week_37,7,7,7,921,60
7,week_38,6,6,6,837,87


In [27]:
summary_total = pd.DataFrame([{
    "recordings": df["recording"].count(),
    "avi_files": df["n_avi_files"].sum(),
    "raw_files": df["n_raw_files"].sum(),
    "avi_GB": df["avi_gb"].sum(),
    "raw_GB": df["raw_gb"].sum(),
}])

summary_total["avi_GB"] = summary_total["avi_GB"].round(0).map(lambda x: f"{int(x):,}")
summary_total["raw_GB"] = summary_total["raw_GB"].round(0).map(lambda x: f"{int(x):,}")

summary_total

,recordings,avi_files,raw_files,avi_GB,raw_GB
0,65,65,65,"10,741",783


## Processed data stats

In [2]:
ROOT = Path(r"//172.16.222.2/tmp/Nora/traffic_flow_data")

In [9]:
rows = []

for week_dir in ROOT.glob("week_*"):
    if not week_dir.is_dir():
        continue

    for box_dir in week_dir.iterdir():
        if not box_dir.is_dir():
            continue

        box_name = box_dir.name

        for rec_dir in box_dir.iterdir():
            if not rec_dir.is_dir():
                continue
            
            for data_dir in rec_dir.iterdir():
                if not data_dir.is_dir():
                    continue

                if data_dir.name == "labels":
                    counter = 0
                    for labels_dir in data_dir.iterdir():
                        counter += 1

                    rows.append({
                        "week": week_dir.name,
                        "box": box_name,
                        "recording": rec_dir.name,
                        "n_labels_batches": counter
                    })

df = pd.DataFrame(rows)

In [ ]:
df[]

,week,box,recording,n_labels_batches
0,week_31,box_1,2024_07_31_16_54_30_recordings,171
1,week_31,box_1,2024_08_01_16_26_38_recordings,169
2,week_31,box_2,2024_07_30_16_23_44_recordings,187
3,week_31,box_2,2024_07_31_17_03_41_recordings,167
4,week_31,box_2,2024_08_01_16_40_03_recordings,162
...,...,...,...,...
62,week_38,box_2,2024_09_24_14_23_34_recordings,180
63,week_38,box_2,2024_09_25_15_54_54_recordings,124
64,week_38,box_3,2024_09_24_14_22_25_recordings,180
65,week_38,box_3,2024_09_25_15_56_11_recordings,114


In [28]:
summary_week = (
    df.groupby("week", as_index=False)
      .agg(
          recordings=("recording", "count"),
          recording_minutes=("n_labels_batches", "sum"),
          recording_hours=("n_labels_batches", lambda x: round(x.sum() / 60, 2))
      )
)

total_row = pd.DataFrame([{
    "week": "Total",
    "recordings": summary_week["recordings"].sum(),
    "recording_minutes": summary_week["recording_minutes"].sum(),
    "recording_hours": summary_week["recording_hours"].sum().round(2)
}])

summary_week = pd.concat([summary_week, total_row], ignore_index=True)
summary_week["recording_minutes"] = (summary_week["recording_minutes"].map(lambda x: f"{x:,}".replace(",", " ")))

In [29]:
summary_week

,week,recordings,recording_minutes,recording_hours
0,week_31,8,1 395,23.25
1,week_32,8,1 394,23.23
2,week_33,8,1 202,20.03
3,week_34,9,1 462,24.37
4,week_35,12,1 632,27.20
5,week_36,9,1 441,24.02
6,week_37,7,885,14.75
7,week_38,6,755,12.58
8,Total,67,10 166,169.43


In [4]:
print(f"Average recording time: {round(169.43/67,2)} hours")

Average recording time: 2.53 hours
